In [ ]:
import numpy as np

rng = np.random.default_rng()

In [ ]:
ZERO = np.array([1, 0], dtype=complex)
ONE = np.array([0, 1], dtype=complex)
PLUS = (ZERO + ONE) / np.sqrt(2)
MINUS = (ZERO - ONE) / np.sqrt(2)


def prepare_pair(b, c0, c1):
    z_states = (ZERO, ONE)
    x_states = (PLUS, MINUS)

    if b == 0:
        # First qubit is Z-encoded; second is X-encoded.
        return z_states[c0].copy(), x_states[c1].copy()

    if b == 1:
        # First qubit is X-encoded; second is Z-encoded.
        return x_states[c0].copy(), z_states[c1].copy()

    raise ValueError("b must be either 0 or 1")

In [ ]:
Z_PROJ_0 = np.array([[1, 0], [0, 0]], dtype=complex)
Z_PROJ_1 = np.array([[0, 0], [0, 1]], dtype=complex)

X_PROJ_PLUS = 0.5 * np.array([[1, 1], [1, 1]], dtype=complex)
X_PROJ_MINUS = 0.5 * np.array([[1, -1], [-1, 1]], dtype=complex)


def measure_qubit_in_place(qubit, proj_0, proj_1):
    """
    Measure `qubit` using the two projectors and collapse the original
    NumPy array in place.

    Returns the measurement outcome, either 0 or 1.
    """
    prob_0 = float(np.real(np.vdot(qubit, proj_0 @ qubit)))
    prob_0 = np.clip(prob_0, 0.0, 1.0)

    outcome = 0 if rng.random() < prob_0 else 1
    projector = proj_0 if outcome == 0 else proj_1

    collapsed_state = projector @ qubit
    collapsed_norm = np.linalg.norm(collapsed_state)

    if collapsed_norm == 0:
        raise RuntimeError("Numerical error produced a zero collapsed state.")

    # # This changes the qubit in the token outside this function.
    qubit[...] = collapsed_state / collapsed_norm

    return outcome


def answer_challenge(pair, challenge):
    if challenge == "ZZ":
        bit0 = measure_qubit_in_place(
            pair[0], Z_PROJ_0, Z_PROJ_1)
        bit1 = measure_qubit_in_place(
            pair[1], Z_PROJ_0, Z_PROJ_1)

    elif challenge == "XX":
        bit0 = measure_qubit_in_place(
            pair[0], X_PROJ_PLUS, X_PROJ_MINUS
        )
        bit1 = measure_qubit_in_place(
            pair[1], X_PROJ_PLUS, X_PROJ_MINUS
        )

    else:
        raise ValueError("challenge must be 'ZZ' or 'XX'.")

    return bit0, bit1


In [ ]:
def verifier(secret, token):
  count = 0
  N = len(secret)
  print(secret)
  print(token)
  for i in range(N):
    b, c0, c1 = secret[i]
    #create random challenge
    rnd = rng.integers(0,2)
    if rnd == 0:
      #ZZ
      (x0,x1) = answer_challenge(token[i], 'ZZ')
      if b == 0:
        if x0 != c0:
          count += 0
        else:
          count += 1
      if b == 1:
        if x1 != c1:
          count += 0
        else:
          count += 1
    if rnd == 1:
      (x0,x1) = answer_challenge(token[i], 'XX')
      if b == 0:
        if x1 != c1:
          count += 0
        else:
          count += 1
      if b == 1:
        if x0 != c0:
          count += 0
        else:
          count += 1
  return count /N


In [ ]:
def attacker1(secret, token):
  N = len(secret)
  #verifier 1
  count1 = 0
  for i in range(N):
    b, c0, c1 = secret[i]
    #create random challenge
    rnd = rng.integers(0,2)
    if rnd == 0:
      #ZZ
      (x0,x1) = (rng.integers(0,2), rng.integers(0,2))
      if b == 0:
        if x0 != c0:
          count1 += 0
        else:
          count1 += 1
      if b == 1:
        if x1 != c1:
          count1 += 0
        else:
          count1 += 1
    if rnd == 1:
      (x0,x1) = answer_challenge(token[i], 'XX')
      if b == 0:
        if x1 != c1:
          count1 += 0
        else:
          count1 += 1
      if b == 1:
        if x0 != c0:
          count1 += 0
        else:
          count1 += 1
  print(count1/N)

  #vertifier 2
  count2 = 0
  for i in range(N):
    b, c0, c1 = secret[i]
    #create random challenge
    rnd = rng.integers(0,2)
    if rnd == 0:
      #ZZ
      (x0,x1) = (rng.integers(0,2), rng.integers(0,2))
      if b == 0:
        if x0 != c0:
          count2 += 0
        else:
          count2 += 1
      if b == 1:
        if x1 != c1:
          count2 += 0
        else:
          count2 += 1
    if rnd == 1:
      (x0,x1) = (rng.integers(0,2), rng.integers(0,2))
      if b == 0:
        if x1 != c1:
          count2 += 0
        else:
          count2 += 1
      if b == 1:
        if x0 != c0:
          count2 += 0
        else:
          count2 += 1
  print(count2/N)


In [ ]:
def attacker2(secret, token):
  N = len(secret)
  #verifier 1
  count1 = 0
  for i in range(N):
    b, c0, c1 = secret[i]
    #create random challenge
    rnd = rng.integers(0,2)
    if rnd == 0:
      #ZZ
      (x0,x1) = answer_challenge(token[i], 'ZZ')
      if b == 0:
        if x0 != c0:
          count1 += 0
        else:
          count1 += 1
      if b == 1:
        if x1 != c1:
          count1 += 0
        else:
          count1 += 1
    if rnd == 1:
      (x0,x1) = answer_challenge(token[i], 'XX')
      if b == 0:
        if x1 != c1:
          count1 += 0
        else:
          count1 += 1
      if b == 1:
        if x0 != c0:
          count1 += 0
        else:
          count1 += 1
  print(count1/N)

  #vertifier 2
  count2 = 0
  for i in range(N):
    b, c0, c1 = secret[i]
    #create random challenge
    rnd = rng.integers(0,2)
    if rnd == 0:
      #ZZ
      (x0,x1) = answer_challenge(token[i], 'ZZ')
      if b == 0:
        if x0 != c0:
          count2 += 0
        else:
          count2 += 1
      if b == 1:
        if x1 != c1:
          count2 += 0
        else:
          count2 += 1
    if rnd == 1:
      (x0,x1) = answer_challenge(token[i], 'XX')
      if b == 0:
        if x1 != c1:
          count2 += 0
        else:
          count2 += 1
      if b == 1:
        if x0 != c0:
          count2 += 0
        else:
          count2 += 1
  print(count2/N)

In [ ]:
V_PROJ_0 = (1/2)*np.array([[(1+1/np.sqrt(2))**2, 1/np.sqrt(2)*(1+1/np.sqrt(2))],
                          [1/np.sqrt(2)*(1+1/np.sqrt(2)), 1/2]], dtype=complex)
V_PROJ_1 = (1/2)*np.array([[1/2, 1/np.sqrt(2)*(1-1/np.sqrt(2))],
                          [1/np.sqrt(2)*(1-1/np.sqrt(2)),(1-1/np.sqrt(2))**2]],
                          dtype=complex)


def answer_challenge_attack3(pair, challenge):
    bit0 = measure_qubit_in_place(
        pair[0], V_PROJ_0, V_PROJ_1)
    bit1 = measure_qubit_in_place(
        pair[1], V_PROJ_0, V_PROJ_1)
    return bit0, bit1


def attacker3(secret, token):
  N = len(secret)
  count1 = 0
  count2 = 0
  count_both = 0
  #verifier 1
  for i in range(N):
    b, c0, c1 = secret[i]
    first_correct = False
    #create random challenge
    rnd = rng.integers(0,2)
    (x0,x1) = answer_challenge_attack3(token[i], 'ZZ')
    if rnd == 0:
      #ZZ
      if b == 0:
        if x0 != c0:
          count1 += 0
        else:
          count1 += 1
          first_correct = True
      if b == 1:
        if x1 != c1:
          count1 += 0
        else:
          count1 += 1
          first_correct = True
    if rnd == 1:
      if b == 0:
        if x1 != c1:
          count1 += 0
        else:
          count1 += 1
          first_correct = True
      if b == 1:
        if x0 != c0:
          count1 += 0
        else:
          count1 += 1
          first_correct = True

    #vertifier 2
    b, c0, c1 = secret[i]
    #create random challenge
    rnd = rng.integers(0,2)
    if rnd == 0:
      #ZZ
      if b == 0:
        if x0 != c0:
          count2 += 0
        else:
          count2 += 1
          if first_correct:
            count_both +=1
      if b == 1:
        if x1 != c1:
          count2 += 0
        else:
          count2 += 1
          if first_correct:
            count_both +=1
    if rnd == 1:
      if b == 0:
        if x1 != c1:
          count2 += 0
        else:
          count2 += 1
          if first_correct:
            count_both +=1
      if b == 1:
        if x0 != c0:
          count2 += 0
        else:
          count2 += 1
          if first_correct:
            count_both +=1
  print(count1/N)
  print(count2/N)
  print(count_both/N)

In [ ]:
N = 10000
token = []
secret = []
for _ in range(N):
  b = int(rng.integers(0,2))
  c0 = int(rng.integers(0,2))
  c1 = int(rng.integers(0,2))
  token.append(prepare_pair(b,c0,c1))
  secret.append((b,c0,c1))


#print(verifier(secret, token))
attacker2(secret, token)

#bank keeps secret and distributes token to user


1.0
0.7432
